#                               Genome Assembly Tutorial - Primary Analysis

## Step 1: Primary Analysis & Quality Control
**Input:** Raw Illumina paired-end reads (`3RR_illumina_R1/R2.fastq.gz`) 
and raw Nanopore reads (`3RR_nanopore.fastq.gz`)  
**Output:** Trimmed/merged reads in `02-primary/`, Kraken2 taxonomy reports  
**Tools:** FastQC v0.12.1, NanoPlot v1.42.0, fastp v1.0.1, Kraken2 v2.1.6, MultiQC  
**Key parameters:** Illumina: avg quality ≥28, min length ≥101 bp; 
Nanopore: avg quality ≥16, min length ≥1000 bp  
**Reference:** Materials & Methods Section 4.1 — Nebli et al. (2025)

## 1. Configuration and Setup

### Environment Variables

In [ ]:
export SN=3RR
export NCPUS=12

## 2. Software dependencies

### Apptainer Aliases

In [ ]:
alias fastqc="apptainer run docker://staphb/fastqc fastqc"
alias NanoPlot="apptainer run docker://staphb/nanoplot NanoPlot"
alias fastp="apptainer run docker://staphb/fastp fastp"
alias multiqc="apptainer run docker://staphb/multiqc multiqc"
alias seqtk="apptainer run docker://staphb/seqtk seqtk"
alias kraken2="apptainer run docker://staphb/kraken2 kraken2"

## 3.Directory Structure Setup

In [ ]:
# Navigate to working directory and create structure
mkdir -p 02-primary/{trimmed,merged,reports,kraken2} # creates kraken2  merged  reports  trimmed within 02-primary
mkdir -p 02-primary/{fastQC,nanoplot}/{raw,processed} #it creates fastQC and nanoplot and within each one it creates two fodders raw and processed
mkdir -p 02-primary/trimmed/sub #it creates a sub folder within the trimmed folder 

## 4.Initial Quality Control Assessment

### Illumina Data QC with FastQC

In [ ]:
unalias fastqc
# Run FastQC on raw Illumina data
fastqc --threads $NCPUS --memory 8000 \
  --extract --outdir 02-primary/fastQC/raw/ \
  01-rawdata/${SN}_illumina_R1.fastq.gz 01-rawdata/${SN}_illumina_R2.fastq.gz

### Examine Key FastQC metrics

### Nanopore Data QC with NanoPlot

In [ ]:
# Run NanoPlot on raw Nanopore data
mkdir -p 02-primary/nanoplot/
NanoPlot --fastq 01-rawdata/${SN}_nanopore.fastq.gz \
  --outdir 02-primary/nanoplot/raw/ --info_in_report \
  --loglength --threads $NCPUS

## Preprocessing: Filtering and Trimming

### Illumina Data Preprocessing with fastp

In [ ]:
alias fastp="apptainer run docker://staphb/fastp fastp"

In [ ]:
export opt=A

fastp \
  -i 01-rawdata/${SN}_illumina_R1.fastq.gz \
  -I 01-rawdata/${SN}_illumina_R2.fastq.gz \
  -o 02-primary/trimmed/${SN}-${opt}-illumina_R1.fastq.gz \
  -O 02-primary/trimmed/${SN}-${opt}-illumina_R2.fastq.gz \
  --adapter_fasta /home/fbouzid/SN/01-rawdata/fastqc_adapter.fasta \
  --trim_poly_g --trim_poly_x \
  --average_qual 28 --length_required 101 \
  --html 02-primary/trimmed/${SN}-${opt}-illumina_fastp.html \
  --cut_right_window_size 4 --cut_right_mean_quality 20 --cut_right \
  --cut_tail_window_size 4 --cut_tail_mean_quality 25 --cut_tail \
  --thread $NCPUS


## Nanopore Data Preprocessing

In [ ]:
# Run fastp for Nanopore data (quality and length filtering)
fastp \
  -i 01-rawdata/${SN}_nanopore.fastq.gz -o 02-primary/trimmed/${SN}-nanopore.fastq.gz \
  --disable_adapter_trimming \
  --average_qual 16 --length_required 1000 \
  --html 02-primary/trimmed/${SN}-nanopore_fastp.html \
  --thread $NCPUS

# Read Merging (Illumina Only)

In [ ]:
# Merge overlapping Illumina read pairs (Option A)
# always change code to specify path to adapter file
export opt=A
fastp \
  --merge --correction --merged_out 02-primary/merged/${SN}-${opt}-illumina.fastq.gz \
  --disable_quality_filtering \
  -i 02-primary/trimmed/${SN}-${opt}-illumina_R1.fastq.gz \
  -I 02-primary/trimmed/${SN}-${opt}-illumina_R2.fastq.gz \
  -o 02-primary/merged/${SN}-${opt}-illumina_R1.fastq.gz \
  -O 02-primary/merged/${SN}-${opt}-illumina_R2.fastq.gz \
  --adapter_fasta /home/fbouzid/SN/01-rawdata/fastqc_adapter.fasta \
  --trim_poly_g --trim_poly_x \
  --html merged/${SN}-${opt}-illumina_merge_fastp.html \
  --thread $NCPUS

# Post-Processing Quality Control

## Post-Processing QC for Illumina Data

In [ ]:
fastqc --threads $NCPUS --memory 8000 \
  --extract --outdir 02-primary/fastQC/processed \
  02-primary/merged/sample10-A-illumina*.fastq.gz

## Post-Processing QC for Nanopore Data

In [ ]:
NanoPlot --fastq 02-primary/trimmed/${SN}-nanopore.fastq.gz \
  --outdir 02-primary/nanoplot/processed --info_in_report \
  --loglength --threads $NCPUS

# Taxonomic Classification with Kraken2

# Kraken2 Classification

In [ ]:

# Create subsamples of trimmed Illumina data (100k reads each)
# Set seed for reproducible subsampling
export SEED=42
export opt=C
export NCPUS=64


gunzip -c ${SN}-${opt}-illumina_R1_entire.fastq.gz > ${SN}-${opt}-illumina_R1_entire.fastq
gunzip -c ${SN}-${opt}-illumina_R2_entire.fastq.gz > ${SN}-${opt}-illumina_R2_entire.fastq

# Run Kraken2 on subsampled datasets (faster analysis)
export SEED=42

apptainer exec \
  --bind  /home/kraken2/:/kraken2db \
  docker://staphb/kraken2 \
  kraken2 \
  --db /kraken2db \
  --threads $NCPUS \
  --paired \
  --output 02-primary/kraken2/${SN}-${opt}-illumina_sub_entire_db.kraken \
  --report 02-primary/kraken2/${SN}-${opt}-illumina_sub_entire_db.report \
  02-primary/trimmed/sub/sample10-C-illumina_R1_entire.fastq \
  02-primary/trimmed/sub/sample10-C-illumina_R2_entire.fastq

#  Comprehensive Reporting with MultiQC

In [ ]:
alias multiqc="apptainer run docker://staphb/multiqc multiqc"

In [ ]:
multiqc 02-primary/ --outdir 02-primary/reports/ --dirs